# OBIA Feature Extraction with Sentinel-2 Imagery


This notebook performs object-based feature extraction from Sentinel-2 imagery using the `nickyspatial` library.

Steps included:
1. Load Sentinel-2 bands (Red and NIR)
2. Compute NDVI
3. Segment the image using Quickshift
4. Compute zonal statistics (mean NDVI)
5. Extract geometry features (area, perimeter, compactness)
6. Compute GLCM texture features (contrast)
7. Save the results as GeoJSON and visualize
    

In [3]:
import nickyspatial
print(dir(nickyspatial))


['EnclosedByRuleSet', 'Layer', 'LayerManager', 'MergeRuleSet', 'Rule', 'RuleSet', 'SlicSegmentation', 'SupervisedClassifier', 'TouchedByRuleSet', '__author__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'attach_area_stats', 'attach_basic_stats', 'attach_class_distribution', 'attach_count', 'attach_ndvi', 'attach_neighbor_stats', 'attach_shape_metrics', 'attach_spectral_indices', 'core', 'create_sample_data', 'enhance_contrast', 'filters', 'io', 'layer_to_raster', 'layer_to_vector', 'merge_small_segments', 'plot_classification', 'plot_comparison', 'plot_histogram', 'plot_layer', 'plot_layer_interactive', 'plot_layer_interactive_plotly', 'plot_sample', 'plot_statistics', 'read_raster', 'read_vector', 'select_by_area', 'smooth_boundaries', 'spectral_filter', 'stats', 'utils', 'viz', 'write_raster', 'write_vector']


In [1]:

import rasterio
import numpy as np
import geopandas as gpd
from nickyspatial.segmentation import quickshift_segmentation
from nickyspatial.zonal import zonal_stats_table
# from skimage.feature import greycomatrix, greycoprops
# from shapely.geometry import shape
import matplotlib.pyplot as plt


ModuleNotFoundError: No module named 'nickyspatial.segmentation'

## Load Sentinel-2 Red and NIR bands

In [ ]:

# Load Sentinel-2 bands from a multiband TIFF
# Replace 'sentinel2.tif' with your actual file path

multiband_path = "sentinel2.tif"

with rasterio.open(multiband_path) as src:
    print(f"Number of bands: {src.count}")
    
    # Read Band 4 (Red) and Band 8 (NIR)
    red = src.read(4).astype("float32")  # Band 4: Red
    nir = src.read(8).astype("float32")  # Band 8: NIR
    
    profile = src.profile
    transform = src.transform


## Compute NDVI

In [ ]:

ndvi = (nir - red) / (nir + red)
ndvi = np.clip(ndvi, -1, 1)

plt.imshow(ndvi, cmap='RdYlGn')
plt.title("NDVI")
plt.colorbar()
plt.show()


## Segment NDVI image using Quickshift

In [ ]:

segments = quickshift_segmentation(ndvi, kernel_size=3, max_dist=6, ratio=0.5)

plt.imshow(segments, cmap='tab20')
plt.title("Segmented Image")
plt.colorbar()
plt.show()


## Compute Zonal Statistics (mean NDVI per object)

In [ ]:

features = {'mean_ndvi': ndvi}
gdf = zonal_stats_table(segments, features, transform=transform)


## Compute Texture Features (GLCM Contrast from NDVI)

In [ ]:

ndvi_int = ((ndvi + 1) * 127.5).astype('uint8')  # Scale to 0–255
glcm = greycomatrix(ndvi_int, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
contrast = greycoprops(glcm, 'contrast')[0, 0]
gdf['glcm_contrast'] = contrast  # Apply global contrast value (can be per object later)


## Add Geometry Features

In [ ]:

gdf['area'] = gdf.geometry.area
gdf['perimeter'] = gdf.geometry.length
gdf['compactness'] = (4 * np.pi * gdf['area']) / (gdf['perimeter'] ** 2)


## Save as GeoJSON

In [ ]:

gdf.to_file("obia_features.geojson", driver="GeoJSON")


## Visualize Results

In [ ]:

gdf.plot(column='mean_ndvi', cmap='YlGn', legend=True)
plt.title("Mean NDVI per Object")
plt.show()
